In [0]:
from pyspark.sql.functions import col, when, sum as spark_sum

In [0]:
accounts_df = spark.read.csv("/Volumes/azuredatabricks0811/default/bronze/accounts.csv", header=True, inferSchema=True)
products_df = spark.read.csv("/Volumes/azuredatabricks0811/default/bronze/products.csv", header=True, inferSchema=True)
sales_pipeline_df = spark.read.csv("/Volumes/azuredatabricks0811/default/bronze/sales_pipeline.csv", header=True, inferSchema=True)
sales_teams_df = spark.read.csv("/Volumes/azuredatabricks0811/default/bronze/sales_teams.csv", header=True, inferSchema=True)
data_dictionary_df = spark.read.csv("/Volumes/azuredatabricks0811/default/bronze/data_dictionary.csv", header=True, inferSchema=True)

In [0]:
accounts_df.show(5)

+----------------+---------+----------------+-------+---------+---------------+-------------+
|         account|   sector|year_established|revenue|employees|office_location|subsidiary_of|
+----------------+---------+----------------+-------+---------+---------------+-------------+
|Acme Corporation|technolgy|            1996|1100.04|     2822|  United States|         NULL|
|      Betasoloin|  medical|            1999| 251.41|      495|  United States|         NULL|
|        Betatech|  medical|            1986| 647.18|     1185|          Kenya|         NULL|
|      Bioholding|  medical|            2012| 587.34|     1356|     Philipines|         NULL|
|         Bioplex|  medical|            1991| 326.82|     1016|  United States|         NULL|
+----------------+---------+----------------+-------+---------+---------------+-------------+
only showing top 5 rows


In [0]:
print(accounts_df.columns)
print(data_dictionary_df.columns)
print(products_df.columns)
print(sales_pipeline_df.columns)
print(sales_teams_df.columns)

['account', 'sector', 'year_established', 'revenue', 'employees', 'office_location', 'subsidiary_of']
['Table', 'Field', 'Description']
['product', 'series', 'sales_price']
['opportunity_id', 'sales_agent', 'product', 'account', 'deal_stage', 'engage_date', 'close_date', 'close_value']
['sales_agent', 'manager', 'regional_office']


In [0]:
accounts_df = accounts_df.withColumnRenamed("subsidiary_of", "parent_company")
data_dictionary_df = (data_dictionary_df
    .withColumnRenamed("Table", "table")
    .withColumnRenamed("Field", "field")
    .withColumnRenamed("Description", "description"))

accounts_df.show(2)
data_dictionary_df.show(2)


+----------------+---------+----------------+-------+---------+---------------+--------------+
|         account|   sector|year_established|revenue|employees|office_location|parent_company|
+----------------+---------+----------------+-------+---------+---------------+--------------+
|Acme Corporation|technolgy|            1996|1100.04|     2822|  United States|          NULL|
|      Betasoloin|  medical|            1999| 251.41|      495|  United States|          NULL|
+----------------+---------+----------------+-------+---------+---------------+--------------+
only showing top 2 rows
+--------+-------+------------+
|   table|  field| description|
+--------+-------+------------+
|accounts|account|Company name|
|accounts| sector|    Industry|
+--------+-------+------------+
only showing top 2 rows


In [0]:
def null_counts(df):
    return df.select([spark_sum(when(col(c).isNull(), 1).otherwise(0)).alias(c) for c in df.columns])

null_counts(accounts_df).display()
null_counts(sales_pipeline_df).display()
null_counts(products_df).display()
null_counts(sales_teams_df).display()


account,sector,year_established,revenue,employees,office_location,parent_company
0,0,0,0,0,0,70


opportunity_id,sales_agent,product,account,deal_stage,engage_date,close_date,close_value
0,0,0,1425,0,500,2089,2089


product,series,sales_price
0,0,0


sales_agent,manager,regional_office
0,0,0


In [0]:
accounts_df = accounts_df.fillna({"parent_company": "Independent"})
sales_pipeline_df = sales_pipeline_df.fillna({"account": "unknown"})


In [0]:
# Data Quality Checks:
def data_quality_check(df, checks, table_name):
    print(f"\nRunning data quality checks for: {table_name}")
    all_passed = True
    for description, passed in checks:
        status = "PASS" if passed else "FAIL"
        print(f"  [{status}] {description}")
        if not passed:
            all_passed = False
    if not all_passed:
        raise ValueError(f"Data quality checks FAILED for {table_name}. Halting before Silver write.")
    print(f"All checks passed for {table_name}. Cleared for Silver.\n")


accounts_checks = [
    ("No null account names", accounts_df.filter(col("account").isNull()).count() == 0),
    ("No null parent_company values (after fillna)", accounts_df.filter(col("parent_company").isNull()).count() == 0),
    ("Revenue is never negative", accounts_df.filter(col("revenue") < 0).count() == 0),
    ("Employees is never negative", accounts_df.filter(col("employees") < 0).count() == 0),
    ("year_established is a reasonable year (1800-2026)",
     accounts_df.filter((col("year_established") < 1800) | (col("year_established") > 2026)).count() == 0),
    ("No fully duplicate rows", accounts_df.count() == accounts_df.dropDuplicates().count()),
]
data_quality_check(accounts_df, accounts_checks, "accounts")


sales_pipeline_checks = [
    ("No null opportunity_id", sales_pipeline_df.filter(col("opportunity_id").isNull()).count() == 0),
    ("No null account values (after fillna)", sales_pipeline_df.filter(col("account").isNull()).count() == 0),
    ("close_value is never negative", sales_pipeline_df.filter(col("close_value") < 0).count() == 0),
    ("deal_stage only contains expected values",
     sales_pipeline_df.filter(~col("deal_stage").isin(["Won", "Lost", "Engaging", "Prospecting"])).count() == 0),
    ("No fully duplicate rows", sales_pipeline_df.count() == sales_pipeline_df.dropDuplicates().count()),
]
data_quality_check(sales_pipeline_df, sales_pipeline_checks, "sales_pipeline")


products_checks = [
    ("No null product names", products_df.filter(col("product").isNull()).count() == 0),
    ("sales_price is never negative", products_df.filter(col("sales_price") < 0).count() == 0),
]
data_quality_check(products_df, products_checks, "products")


sales_teams_checks = [
    ("No null sales_agent values", sales_teams_df.filter(col("sales_agent").isNull()).count() == 0),
    ("No fully duplicate rows", sales_teams_df.count() == sales_teams_df.dropDuplicates().count()),
]
data_quality_check(sales_teams_df, sales_teams_checks, "sales_teams")



Running data quality checks for: accounts
  [PASS] No null account names
  [PASS] No null parent_company values (after fillna)
  [PASS] Revenue is never negative
  [PASS] Employees is never negative
  [PASS] year_established is a reasonable year (1800-2026)
  [PASS] No fully duplicate rows
All checks passed for accounts. Cleared for Silver.


Running data quality checks for: sales_pipeline
  [PASS] No null opportunity_id
  [PASS] No null account values (after fillna)
  [PASS] close_value is never negative
  [PASS] deal_stage only contains expected values
  [PASS] No fully duplicate rows
All checks passed for sales_pipeline. Cleared for Silver.


Running data quality checks for: products
  [PASS] No null product names
  [PASS] sales_price is never negative
All checks passed for products. Cleared for Silver.


Running data quality checks for: sales_teams
  [PASS] No null sales_agent values
  [PASS] No fully duplicate rows
All checks passed for sales_teams. Cleared for Silver.



In [0]:
null_counts(accounts_df).display()

account,sector,year_established,revenue,employees,office_location,parent_company
0,0,0,0,0,0,0


In [0]:
accounts_df.write.option("header", "true").mode("overwrite").csv("/Volumes/azuredatabricks0811/default/silver/accounts")
sales_pipeline_df.write.option("header", "true").mode("overwrite").csv("/Volumes/azuredatabricks0811/default/silver/sales_pipeline")
sales_teams_df.write.option("header", "true").mode("overwrite").csv("/Volumes/azuredatabricks0811/default/silver/sales_teams")
products_df.write.option("header", "true").mode("overwrite").csv("/Volumes/azuredatabricks0811/default/silver/products")
data_dictionary_df.write.option("header", "true").mode("overwrite").csv("/Volumes/azuredatabricks0811/default/silver/data_dictionary")

print("All tables written to Silver successfully.")

All tables written to Silver successfully.


In [0]:
sales_pipeline_df.select("deal_stage").distinct().show()

+-----------+
| deal_stage|
+-----------+
|        Won|
|       Lost|
|Prospecting|
|   Engaging|
+-----------+

